# Template Quest Notebook

In [1]:
# Import neccessary modules, add to this cell as needed
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

## Part 1: Load the Sample Dataset

In [2]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

In [3]:
df_bronze = spark.read.json("./data/sysmon_spearphish_cribl.json")

In [4]:
raw_schema = "Image string, UserID string, QueryName string, QueryStatus decimal, QueryResults string, SystemTime string, ProcessId string, Channel string"

df_silver = (df_bronze
    .select(
        F.to_timestamp(F.col("_time")).alias("_time"),
        "Computer",
        "EventCode",
        "User",
        F.from_json(F.col("_raw"), raw_schema).alias("_parsed"),
        "_raw",
    )
    .select("*", "_parsed.*")
    .drop("_parsed")
)
df_silver.createOrReplaceTempView("sysmon_silver")

In [5]:
# PySpark Example
df_silver.filter("EventCode == '22'").limit(5).show()

+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|               _time|            Computer|EventCode|User|                _raw|               Image|  UserID|           QueryName|QueryStatus|        QueryResults|          SystemTime|ProcessId|             Channel|
+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Temp\OfficeSet...|S-1-5-18|      ecs.office.com|          0|type:  5 ecs.offi...|'2023-01-27T11:22...|     6048|Microsoft-Windows...|
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Windows\System...|S-1-5-18|f.c2r.ts.cdn.offi...|      

In [6]:
# SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver
WHERE EventCode == 22
LIMIT 100
""").show()

+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|               _time|            Computer|EventCode|User|                _raw|               Image|  UserID|           QueryName|QueryStatus|        QueryResults|          SystemTime|ProcessId|             Channel|
+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Temp\OfficeSet...|S-1-5-18|      ecs.office.com|          0|type:  5 ecs.offi...|'2023-01-27T11:22...|     6048|Microsoft-Windows...|
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Windows\System...|S-1-5-18|f.c2r.ts.cdn.offi...|      

## Part 2: Detection Engineering

In [7]:
# prove hypothesis
spark.sql("""
SELECT _time, Computer, EventCode, User, Image, UserID, QueryName, QueryResults
FROM sysmon_silver
WHERE EventCode = 22
AND LOWER(Image) LIKE "%microsoft office%"
AND LOWER(Image) LIKE "%.exe"
AND NOT(LOWER(QueryName) LIKE "%.office.net" OR LOWER(QueryName) LIKE "%.office.com")
""").show(truncate=False)


+-----------------------+------------------------------+---------+----+-----------------------------------------------------------+--------+-----------------+----------------------------------------+
|_time                  |Computer                      |EventCode|User|Image                                                      |UserID  |QueryName        |QueryResults                            |
+-----------------------+------------------------------+---------+----+-----------------------------------------------------------+--------+-----------------+----------------------------------------+
|2023-01-27 22:30:16.279|win-host-ctus-attack-range-212|22       |NULL|C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|S-1-5-18|www.mediafire.com|::ffff:104.16.54.48;::ffff:104.16.53.48;|
+-----------------------+------------------------------+---------+----+-----------------------------------------------------------+--------+-----------------+----------------------------------------+


In [8]:
# create new dataframe with result
df_detect = spark.sql("""
SELECT _time, Computer, EventCode, User, Image, UserID, QueryName, QueryResults
FROM sysmon_silver
WHERE EventCode = 22
AND LOWER(Image) LIKE "%microsoft office%"
AND LOWER(Image) LIKE "%.exe"
AND NOT(LOWER(QueryName) LIKE "%.office.net" OR LOWER(QueryName) LIKE "%.office.com")
""")

In [9]:
# display dataframe with DNS queries
spark.sql("""
SELECT
  _time AS `@timestamp`,
  Computer AS `host.name`,
  EventCode AS `event.code`,
  User AS `user.name`,
  Image AS `process.executable`,
  UserID AS `user.id`,
  QueryName AS `dns.query.name`,
  QueryResults AS `dns.query.results`
  FROM sysmon_silver
  WHERE EventCode = 22
""").show(truncate=False)

+-----------------------+------------------------------+----------+---------+------------------------------------------------------------------------------+--------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|@timestamp             |host.name                     |event.code|user.name|process.executable                                                            |user.id |dns.query.name                   |dns.query.results                                                                                                                                                         |
+-----------------------+------------------------------+----------+---------+------------------------------------------------------------------------------+--------+---------------------------------+-----------------------------------------------------------

In [10]:
spark.sql("""
SELECT _time AS `_time`, Computer AS `src`, QueryName AS `query`, QueryResults AS `answer`, 'dns' AS `tag`
FROM sysmon_silver
WHERE EventCode = 22
""").show(truncate=False)


+-----------------------+------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|_time                  |src                           |query                            |answer                                                                                                                                                                    |tag|
+-----------------------+------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|2023-01-27 22:22:35.226|win-host-ctus-attack-range-212|ecs.office.com                   |type:  5 ecs.office.trafficmanager.net;type:  5 s-0005-office.config.skype.com;type:  5 ecs-office.s-0005.s-msed

## Part 3: Additional Steps

### Part 3.1: Normalization

In [11]:
# create normalised dataframe with DNS queries
spark.sql("""
SELECT _time AS `_time`, Computer AS `src`, QueryName AS `query`, QueryResults AS `answer`, 'dns' AS `tag`
FROM sysmon_silver
WHERE EventCode = 22
""").show(truncate=False)

+-----------------------+------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|_time                  |src                           |query                            |answer                                                                                                                                                                    |tag|
+-----------------------+------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|2023-01-27 22:22:35.226|win-host-ctus-attack-range-212|ecs.office.com                   |type:  5 ecs.office.trafficmanager.net;type:  5 s-0005-office.config.skype.com;type:  5 ecs-office.s-0005.s-msed

### Part 3.2: Alert Table

In [12]:
# create new dataframe as alert table
df_alert_table = spark.sql("""
SELECT _time, "Sysmon" AS LogSource, "Potentially malicious webcall intiated by Microsoft Office application" AS Title,"High" AS Severity, Computer, EventCode, User, Image, UserID, QueryName, QueryResults, "Execution" AS MitreTactic, "T1203" AS MitreID, "Exploitation for Client Execution" AS MitreTechnique
FROM sysmon_silver
WHERE EventCode = 22
AND LOWER(Image) LIKE "%microsoft office%"
AND LOWER(Image) LIKE "%.exe"
AND NOT(LOWER(QueryName) LIKE "%.office.net" OR LOWER(QueryName) LIKE "%.office.com")
""")

df_alert_table.show(truncate=False)

+-----------------------+---------+----------------------------------------------------------------------+--------+------------------------------+---------+----+-----------------------------------------------------------+--------+-----------------+----------------------------------------+-----------+-------+---------------------------------+
|_time                  |LogSource|Title                                                                 |Severity|Computer                      |EventCode|User|Image                                                      |UserID  |QueryName        |QueryResults                            |MitreTactic|MitreID|MitreTechnique                   |
+-----------------------+---------+----------------------------------------------------------------------+--------+------------------------------+---------+----+-----------------------------------------------------------+--------+-----------------+----------------------------------------+-----------+-------+---

### Part 3.3: Enrichment

## Summary

Summarize your submission here, comments are helpful to add throughout your code as well.